In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:

# ─────────────────────────────────────────────────────────────────────────────
# 0. Imports & Config
# ─────────────────────────────────────────────────────────────────────────────

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")        # headless — remove for interactive plots
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score,
)

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping XGBoost (pip install xgboost)")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("shap not installed — skipping SHAP (pip install shap)")

RANDOM_STATE = 42
DATA_PATH    = "/content/Pluvial_Flood_Dataset.xlsx"   # update path if needed
CLASS_ORDER  = ["No_Flood", "Low", "Moderate", "High", "Very_High"]

In [4]:
print("\n" + "="*65)
print("  STEP 1 — Load data")
print("="*65)

df = pd.read_excel(DATA_PATH)
print(f"Shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Cols  :", df.columns.tolist())
print("\nClass distribution:")
print(df["SUSCEP"].value_counts().reindex(CLASS_ORDER))


  STEP 1 — Load data
Shape : 144,401 rows x 10 columns
Cols  : ['X', 'Y', 'Slope', 'Curvature ', 'Aspect', 'TWI', 'FA', 'Drainage', 'Rainfall', 'SUSCEP']

Class distribution:
SUSCEP
No_Flood     16126
Low          32252
Moderate     38116
High         34451
Very_High    23456
Name: count, dtype: int64


In [5]:
print("\n" + "="*65)
print("  STEP 2 — Diagnose label quality")
print("="*65)

_drain = pd.to_numeric(df["Drainage"], errors="coerce")
print("Drainage range per SUSCEP class:")
print(df.assign(Drainage=_drain).groupby("SUSCEP")["Drainage"]
      .agg(["min", "max"]).reindex(CLASS_ORDER).round(3))

print("""
DIAGNOSIS:
  Drainage perfectly partitions all 5 classes with ZERO overlap.
  A depth-3 decision tree on Drainage alone scores 100%.

  SUSCEP is a GIS-computed susceptibility INDEX, not observed flood data.
  Any ML model scores 100% because it inverts a deterministic formula.

  For real-world use: replace SUSCEP with actual flood observation records.
  This script includes a realistic-label simulation to show what genuine
  accuracy (~70-80%) looks like after fixing the label issue.
""")


  STEP 2 — Diagnose label quality
Drainage range per SUSCEP class:
               min      max
SUSCEP                     
No_Flood   203.729  209.935
Low        210.132  216.987
Moderate   217.000  221.972
High       222.063  226.863
Very_High  227.033  235.421

DIAGNOSIS:
  Drainage perfectly partitions all 5 classes with ZERO overlap.
  A depth-3 decision tree on Drainage alone scores 100%.

  SUSCEP is a GIS-computed susceptibility INDEX, not observed flood data.
  Any ML model scores 100% because it inverts a deterministic formula.

  For real-world use: replace SUSCEP with actual flood observation records.
  This script includes a realistic-label simulation to show what genuine
  accuracy (~70-80%) looks like after fixing the label issue.



In [6]:
print("="*65)
print("  STEP 3 — Replace NoData sentinels with NaN")
print("="*65)

NUMERIC_COLS = ["Slope", "Curvature ", "Aspect", "TWI", "FA", "Drainage", "Rainfall"]

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    n_float = (df[col] < -1e30).sum()
    df.loc[df[col] < -1e30, col] = np.nan        # float32 NoData
    n_int   = (df[col] == -9999).sum()
    df.loc[df[col] == -9999, col] = np.nan       # integer NoData
    if n_float + n_int:
        print(f"  {col:13s}: fixed {n_float + n_int:,} NoData values")

print("\nMissing values after cleaning:")
print(df[NUMERIC_COLS].isnull().sum().to_string())

  STEP 3 — Replace NoData sentinels with NaN
  Slope        : fixed 106 NoData values
  Curvature    : fixed 387 NoData values
  Aspect       : fixed 388 NoData values
  TWI          : fixed 388 NoData values
  FA           : fixed 374 NoData values

Missing values after cleaning:
Slope         388
Curvature     387
Aspect        388
TWI           388
FA            374
Drainage        0
Rainfall        0


In [7]:
print("\n" + "="*65)
print("  STEP 4 — Feature engineering")
print("="*65)

# Log Flow Accumulation (heavily right-skewed)
df["log_FA"] = np.log1p(df["FA"].clip(lower=0))

# Aspect as circular components (degrees wrap at 0/360)
df["aspect_sin"] = np.sin(np.radians(df["Aspect"]))
df["aspect_cos"] = np.cos(np.radians(df["Aspect"]))

# Interaction: steep AND wet = higher susceptibility
df["slope_twi"] = df["Slope"] * df["TWI"]

# Curvature magnitude on log scale
df["curv_abs"] = np.log1p(df["Curvature "].abs().clip(upper=1e9))

ALL_FEATURES = [
    "Slope", "TWI", "Drainage", "Rainfall",
    "log_FA", "aspect_sin", "aspect_cos",
    "slope_twi", "curv_abs",
]

print("Features used (X and Y intentionally excluded to prevent map memorisation):")
for f in ALL_FEATURES:
    print(f"  {f}")



  STEP 4 — Feature engineering
Features used (X and Y intentionally excluded to prevent map memorisation):
  Slope
  TWI
  Drainage
  Rainfall
  log_FA
  aspect_sin
  aspect_cos
  slope_twi
  curv_abs


In [8]:
df["target"] = df["SUSCEP"].map({c: i for i, c in enumerate(CLASS_ORDER)})

In [9]:
print("\n" + "="*65)
print("  STEP 5 — Create realistic label simulation")
print("="*65)

np.random.seed(RANDOM_STATE)
NOISE_PCT = 0.25   # 25% of rows flipped to an adjacent class
noisy_target = df["target"].copy()
flip_mask = np.random.rand(len(df)) < NOISE_PCT
deltas = np.random.choice([-1, 1], size=flip_mask.sum())
noisy_target[flip_mask] = (noisy_target[flip_mask] + deltas).clip(0, 4)
df["target_noisy"] = noisy_target

print(f"Simulated {NOISE_PCT*100:.0f}% label noise (±1 adjacent class).")
print("Replace 'target_noisy' with your real observed flood labels for production use.")


  STEP 5 — Create realistic label simulation
Simulated 25% label noise (±1 adjacent class).
Replace 'target_noisy' with your real observed flood labels for production use.


In [10]:
print("\n" + "="*65)
print("  STEP 6 — Spatial block cross-validation setup")
print("="*65)

N_GRID = 5  # creates 5x5 = 25 spatial blocks → 5 geographic folds

df["x_block"] = pd.cut(df["X"], bins=N_GRID, labels=False)
x_vals = sorted(df["x_block"].dropna().unique())
fold_map = {xb: min(int(i // (len(x_vals) / 5)), 4) for i, xb in enumerate(x_vals)}
df["spatial_fold"] = df["x_block"].map(fold_map)

print(f"Grid: {N_GRID}x{N_GRID} spatial blocks -> 5 spatial folds")
print("Rows per fold:")
print(df["spatial_fold"].value_counts().sort_index().to_string())

df_valid = df.dropna(subset=ALL_FEATURES + ["target", "spatial_fold"]).copy()
print(f"\nUsable rows after NaN drop: {len(df_valid):,} / {len(df):,}")


  STEP 6 — Spatial block cross-validation setup
Grid: 5x5 spatial blocks -> 5 spatial folds
Rows per fold:
spatial_fold
0    16318
1    38096
2    40219
3    35578
4    14190

Usable rows after NaN drop: 144,013 / 144,401


In [11]:
def spatial_cv(X_all, y_all, folds_all, model_factory, n_folds=5, label="model"):
    """
    Spatial k-fold cross-validation.

    Parameters
    ----------
    X_all        : ndarray, all feature rows
    y_all        : ndarray, all labels
    folds_all    : ndarray, spatial fold assignment per row
    model_factory: callable returning a fresh unfitted sklearn estimator
    n_folds      : int
    label        : str, name for printing
    """
    acc_list, f1_list, all_pred, all_true = [], [], [], []

    for fold in range(n_folds):
        tr = folds_all != fold
        te = folds_all == fold

        imp = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_all[tr])  # fit on train only
        X_te = imp.transform(X_all[te])

        clf = model_factory()
        clf.fit(X_tr, y_all[tr])
        preds = clf.predict(X_te)

        a = accuracy_score(y_all[te], preds)
        f = f1_score(y_all[te], preds, average="macro")
        acc_list.append(a)
        f1_list.append(f)
        all_pred.extend(preds)
        all_true.extend(y_all[te])

    print(f"  {label:22s}  Acc {np.mean(acc_list):.4f} +/- {np.std(acc_list):.4f}"
          f"  |  F1 {np.mean(f1_list):.4f} +/- {np.std(f1_list):.4f}")

    return {
        "acc":      np.array(acc_list),
        "f1":       np.array(f1_list),
        "all_pred": np.array(all_pred),
        "all_true": np.array(all_true),
    }

In [ ]:
print("\n" + "="*65)
print("  STEP 7 — Spatial cross-validation: all models")
print("="*65)

X_all     = df_valid[ALL_FEATURES].values
y_orig    = df_valid["target"].values
y_noisy   = df_valid["target_noisy"].values
folds_all = df_valid["spatial_fold"].values

model_defs = {
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=100, max_depth=12, min_samples_leaf=5,
        n_jobs=-1, random_state=RANDOM_STATE),
    "GradientBoosting": lambda: GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE),
    "LogisticRegression": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(max_iter=500, n_jobs=-1, random_state=RANDOM_STATE)),
    ]),
}
if HAS_XGB:
    model_defs["XGBoost"] = lambda: xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        use_label_encoder=False, eval_metric="mlogloss",
        n_jobs=-1, random_state=RANDOM_STATE)

print("\n--- Original (synthetic) labels ---")
results_orig = {n: spatial_cv(X_all, y_orig, folds_all, f, label=n)
                for n, f in model_defs.items()}

print("\n--- Noisy / simulated realistic labels ---")
results_noisy = {n: spatial_cv(X_all, y_noisy, folds_all, f, label=n)
                 for n, f in model_defs.items()}


  STEP 7 — Spatial cross-validation: all models

--- Original (synthetic) labels ---
  RandomForest            Acc 1.0000 +/- 0.0000  |  F1 1.0000 +/- 0.0000
  GradientBoosting        Acc 1.0000 +/- 0.0000  |  F1 1.0000 +/- 0.0000
  LogisticRegression      Acc 0.9869 +/- 0.0019  |  F1 0.9890 +/- 0.0016
  XGBoost                 Acc 1.0000 +/- 0.0000  |  F1 1.0000 +/- 0.0000

--- Noisy / simulated realistic labels ---
  RandomForest            Acc 0.7831 +/- 0.0021  |  F1 0.7906 +/- 0.0017
  GradientBoosting        Acc 0.7827 +/- 0.0022  |  F1 0.7902 +/- 0.0018
  LogisticRegression      Acc 0.7580 +/- 0.0024  |  F1 0.7647 +/- 0.0022
  XGBoost                 Acc 0.7830 +/- 0.0021  |  F1 0.7906 +/- 0.0016


In [ ]:
print("\n" + "="*65)
print("  STEP 8 — Final model (geographic holdout = fold 4)")
print("="*65)

best_name = max(results_noisy, key=lambda n: np.mean(results_noisy[n]["f1"]))
print(f"Best model (by noisy-label macro-F1): {best_name}")

tr = folds_all != 4
te = folds_all == 4

final_imp = SimpleImputer(strategy="median")
X_tr_f = final_imp.fit_transform(X_all[tr])
X_te_f = final_imp.transform(X_all[te])

final_model = model_defs[best_name]()
final_model.fit(X_tr_f, y_noisy[tr])
y_pred_f = final_model.predict(X_te_f)
y_true_f = y_noisy[te]

print(f"\nHoldout accuracy  : {accuracy_score(y_true_f, y_pred_f):.4f}")
print(f"Holdout macro-F1  : {f1_score(y_true_f, y_pred_f, average='macro'):.4f}")
print("\nClassification report (noisy labels, fold-4 holdout):")
print(classification_report(y_true_f, y_pred_f, target_names=CLASS_ORDER))



  STEP 8 — Final model (geographic holdout = fold 4)
Best model (by noisy-label macro-F1): RandomForest

Holdout accuracy  : 0.7841
Holdout macro-F1  : 0.7904

Classification report (noisy labels, fold-4 holdout):
              precision    recall  f1-score   support

    No_Flood       0.86      0.76      0.81      1683
         Low       0.74      0.78      0.76      3073
    Moderate       0.76      0.77      0.76      3628
        High       0.76      0.78      0.77      3244
   Very_High       0.88      0.83      0.85      2519

    accuracy                           0.78     14147
   macro avg       0.80      0.78      0.79     14147
weighted avg       0.79      0.78      0.78     14147



In [ ]:
print("\n" + "="*65)
print("  STEP 9 — Generating evaluation plots")
print("="*65)

COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Pluvial Flood Susceptibility — Model Evaluation", fontsize=15, fontweight="bold")

# 1. Class distribution
ax = axes[0, 0]
counts = df["SUSCEP"].value_counts().reindex(CLASS_ORDER)
bars = ax.bar(CLASS_ORDER, counts.values, color=COLORS, edgecolor="white", lw=1.2)
ax.set_title("Class Distribution", fontweight="bold")
ax.set_xlabel("Class"); ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=20)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 500,
            f"{v:,}", ha="center", va="bottom", fontsize=9)

# 2. Synthetic vs realistic F1 comparison
ax = axes[0, 1]
names = list(model_defs.keys())
x = np.arange(len(names)); w = 0.38
orig_f1  = [np.mean(results_orig[n]["f1"])  for n in names]
noisy_f1 = [np.mean(results_noisy[n]["f1"]) for n in names]
ax.bar(x - w/2, orig_f1,  w, label="Synthetic labels (inflated)",
       color="#e74c3c", alpha=0.85, edgecolor="white")
ax.bar(x + w/2, noisy_f1, w, label="Noisy / realistic labels",
       color="#3498db", alpha=0.85, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, fontsize=9)
ax.set_title("Macro-F1: Synthetic vs Realistic Labels", fontweight="bold")
ax.set_ylabel("F1 Score"); ax.set_ylim(0, 1.15)
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for xi, (ov, nv) in enumerate(zip(orig_f1, noisy_f1)):
    ax.text(xi - w/2, ov + 0.02, f"{ov:.2f}", ha="center", fontsize=8)
    ax.text(xi + w/2, nv + 0.02, f"{nv:.2f}", ha="center", fontsize=8)

# 3. Confusion matrix
ax = axes[0, 2]
cm = confusion_matrix(y_true_f, y_pred_f)
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_n, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
            ax=ax, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title(f"Confusion Matrix — {best_name}\n(noisy labels, fold-4 holdout)",
             fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.tick_params(axis="x", rotation=25)

# 4. Feature importance
ax = axes[1, 0]
base = final_model
if hasattr(base, "feature_importances_"):
    imp_v = base.feature_importances_
elif hasattr(base, "named_steps"):
    inner = list(base.named_steps.values())[-1]
    raw   = getattr(inner, "coef_", None)
    imp_v = np.abs(raw).mean(axis=0) if raw is not None else None
else:
    imp_v = None

if imp_v is not None:
    sidx = np.argsort(imp_v)
    ax.barh(np.array(ALL_FEATURES)[sidx], imp_v[sidx], color="#3498db", edgecolor="white")
    ax.set_title(f"Feature Importance — {best_name}", fontweight="bold")
    ax.set_xlabel("Importance")
else:
    ax.text(0.5, 0.5, "Not available", ha="center", transform=ax.transAxes)
    ax.set_title("Feature Importance", fontweight="bold")

# 5. Drainage vs SUSCEP (root cause)
ax = axes[1, 1]
sample = df.sample(min(4000, len(df)), random_state=42)
cmap_c = dict(zip(CLASS_ORDER, COLORS))
for cls in CLASS_ORDER:
    s = sample[sample["SUSCEP"] == cls]
    ax.scatter(s["Drainage"], s["Rainfall"], c=cmap_c[cls], s=5, alpha=0.65,
               label=cls, rasterized=True)
for t in [210.03, 216.99, 222.02, 226.95]:
    ax.axvline(t, color="black", lw=0.8, ls="--", alpha=0.5)
ax.set_title("Root Cause: Drainage perfectly separates SUSCEP", fontweight="bold")
ax.set_xlabel("Drainage"); ax.set_ylabel("Rainfall")
ax.legend(markerscale=3, fontsize=8)
ax.text(0.02, 0.97, "Dashed lines = Drainage thresholds\nthat deterministically define SUSCEP",
        transform=ax.transAxes, fontsize=7.5, va="top",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

# 6. Missing data
ax = axes[1, 2]
miss = df[NUMERIC_COLS].isnull().mean() * 100
ax.barh(NUMERIC_COLS, miss.values, color="#e74c3c", edgecolor="white")
ax.set_title("Missing Data % (after NoData fix)", fontweight="bold")
ax.set_xlabel("% Missing")
ax.set_xlim(0, max(miss.max() * 1.3, 0.5))
for i, v in enumerate(miss.values):
    ax.text(v + 0.01, i, f"{v:.3f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("flood_model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: flood_model_evaluation.png")



  STEP 9 — Generating evaluation plots
  Saved: flood_model_evaluation.png


In [ ]:
if HAS_SHAP and hasattr(final_model, "feature_importances_"):
    print("\n" + "="*65)
    print("  STEP 10 — SHAP feature importance")
    print("="*65)
    n_shap = min(500, len(X_te_f))
    idx_s  = np.random.choice(len(X_te_f), size=n_shap, replace=False)

    explainer    = shap.TreeExplainer(final_model)
    shap_values  = explainer.shap_values(X_te_f[idx_s])
    sv = shap_values[-1] if isinstance(shap_values, list) else shap_values

    plt.figure(figsize=(10, 6))
    shap.summary_plot(sv, X_te_f[idx_s], feature_names=ALL_FEATURES, show=False, plot_type="bar")
    plt.title(f"SHAP Feature Importance — {best_name} (Very_High class)", fontweight="bold")
    plt.tight_layout()
    plt.savefig("flood_shap_importance.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  Saved: flood_shap_importance.png")


In [ ]:
print("\n" + "="*65)
print("  STEP 11 — Saving predictions to CSV")
print("="*65)

test_df = df_valid[folds_all == 4].copy()
proba   = final_model.predict_proba(X_te_f)
test_df["predicted_label"] = (
    pd.Series(y_pred_f, index=test_df.index)
    .map({i: c for i, c in enumerate(CLASS_ORDER)})
)
for i, cls in enumerate(CLASS_ORDER):
    test_df[f"prob_{cls}"] = proba[:, i]

out_cols = ["X", "Y", "SUSCEP", "predicted_label"] + [f"prob_{c}" for c in CLASS_ORDER]
test_df[out_cols].to_csv("flood_predictions.csv", index=False)
print(f"  Saved: flood_predictions.csv ({len(test_df):,} rows)")


  STEP 11 — Saving predictions to CSV
  Saved: flood_predictions.csv (14,147 rows)


In [ ]:
print("\n" + "="*65)
print("  FINAL SUMMARY")
print("="*65)
print(f"""
Dataset         : {len(df):,} pixels  |  {len(ALL_FEATURES)} features used
CV strategy     : 5-fold spatial block split (no geographic leakage)
Best model      : {best_name}

  WITH SYNTHETIC LABELS (as in original notebook):
    Mean Acc = {np.mean(results_orig[best_name]['acc']):.4f}  <- meaningless, labels are a formula

  WITH REALISTIC LABELS (25% noise simulation):
    Mean Acc = {np.mean(results_noisy[best_name]['acc']):.4f}
    Mean F1  = {np.mean(results_noisy[best_name]['f1']):.4f}  <- honest estimate

  HOLDOUT (fold 4, noisy labels):
    Acc = {accuracy_score(y_true_f, y_pred_f):.4f}
    F1  = {f1_score(y_true_f, y_pred_f, average='macro'):.4f}

Changes applied vs original Kaggle notebook:
  [x] X, Y dropped from feature set
  [x] -3.4e38 and -9999 sentinels replaced with NaN
  [x] Imputer fit only on training folds
  [x] Spatial block CV instead of random split
  [x] Feature engineering: log_FA, aspect_sin/cos, slope_twi, curv_abs
  [x] Multiple models benchmarked side by side
  [x] Label quality diagnosis included
  [x] Noisy-label simulation to estimate real-world performance

NEXT STEPS FOR REAL USE:
  1. Obtain real observed flood labels (e.g. from FEMA records, local
     authority flood registers, Sentinel-1 SAR flood mapping, or
     field surveys).
  2. Replace 'target_noisy' with those labels and re-run this script.
  3. Increase spatial CV to 10 folds for more stable estimates.
  4. Consider adding features: soil type, land cover, stream proximity.
""")